In [ ]:
# Framingham Heart Study: 10-Year CHD Prediction

This project predicts 10-year risk of Coronary Heart Disease (CHD) using the Framingham Heart Study dataset.  
It includes data cleaning, exploratory data analysis (EDA), modelling, evaluation, and feature importance analysis with SHAP.


## 1. Load and Inspect Data

In [ ]:
# Library imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("framingham.csv")

# Inspect data
print("Dataset shape:", df.shape)
print("Columns:", df.columns)
df.head()

## 2. Data Cleaning
Convert categorical variables, fill missing values for numerical features, and check dataset integrity.

In [ ]:
# Categorical columns
cat_cols = ["male", "currentSmoker", "BPMeds", "prevalentStroke", 
            "prevalentHyp", "diabetes", "TenYearCHD"]

# Convert to category type
for col in cat_cols:
    df[col] = df[col].astype("category")

# Numerical columns
num_cols = ["cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate",
            "glucose", "education"]

# Fill missing values with median
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill missing BPMeds values with 0
df["BPMeds"].fillna(0, inplace=True)
df["BPMeds"] = df["BPMeds"].astype("category")

# Confirm no missing values remain
print(df.isnull().sum())
df.info()

In [ ]:
#BPMeds had 53 missing values. Fill values with 0
df["BPMeds"].fillna(0, inplace=True)

#Convert to categorical again to be safe
df["BPMeds"] = df["BPMeds"].astype("category")

#Confirm no missing values remain
print(df.isnull().sum())

## 3. Exploratory Data Analysis (EDA)
Visualise the target distribution, correlations, and selected feature histograms.

In [ ]:
# Target distribution
sns.countplot(x="TenYearCHD", data=df)
plt.title("Distribution of 10-Year CHD Outcome")
plt.show()
print(df["TenYearCHD"].value_counts())

# Correlation heatmap
corr = df[num_cols + ["age"]].corr()
plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap of Numerical Features")
plt.show()

# Histograms for key features
features_to_plot = ["age", "BMI", "sysBP", "totChol", "glucose"]
for feature in features_to_plot:
    plt.figure(figsize=(6,4))
    sns.histplot(df[feature], kde=True)
    plt.title(f"Distribution of {feature}")
    plt.show()

## 4. Feature Preparation
Encode categorical features as numeric codes and define target variable.

In [ ]:
# Drop target from features
target = 'TenYearCHD'
features = df.drop(columns=[target])

# Encode categorical features
cat_cols = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke',
            'prevalentHyp', 'diabetes']
for col in cat_cols:
    features[col] = features[col].cat.codes

# Target as numeric
y = df[target].cat.codes

## 5. Train/Test Split and Scaling
Split dataset into training and testing sets and apply standard scaling.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 80/20 split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    features, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Modelling: Logistic Regression, Random Forest, XGBoost
Train three models and store them in a dictionary for iteration.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Define models
lr = LogisticRegression(random_state=42, max_iter=1000)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)

# Dictionary to loop through
models = {"Logistic Regression": lr, "Random Forest": rf, "XGBoost": xgb}

## 7. Model Evaluation
Evaluate models using accuracy, ROC-AUC, confusion matrix, and classification report.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay

results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    results[name] = {'Accuracy': acc, 'ROC-AUC': roc}
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    # Classification Report
    print(f"{name} Classification Report:\n")
    print(classification_report(y_test, y_pred))
    
    # ROC Curve
    RocCurveDisplay.from_estimator(model, X_test_scaled, y_test)
    plt.title(f'{name} - ROC Curve')
    plt.show()

# Model comparison
results_df = pd.DataFrame(results).T
print("Model comparison:\n", results_df)

## 8. Feature Importance and SHAP Analysis
Analyse feature importance using Random Forest and SHAP.

In [ ]:
import shap

# Random Forest feature importance
rf_model = models['Random Forest']
importances = rf_model.feature_importances_
feature_names = X_train.columns

feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp = feat_imp.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(x='Importance', y='Feature', data=feat_imp)
plt.title('Random Forest Feature Importance')
plt.show()

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer(X_test)

# Bar plot
shap.summary_plot(shap_values, X_test, plot_type="bar")

# Beeswarm plot
shap.summary_plot(shap_values, X_test)


# Save results
feat_imp.to_csv('feature_importance_rf.csv', index=False)
y_pred_df = pd.DataFrame({'Actual': y_test, 'Predicted': rf_model.predict(X_test_scaled)})
y_pred_df.to_csv('rf_predictions.csv', index=False)
